# Marker Repo - annotation

In this notebook, clustered h5ad files can be annotated using the MarkerRepo or SCSA.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import markerrepo.parsing as pars
import scanpy as sc

%load_ext autoreload
%autoreload 2

## Settings

Specify path of the cloned repository and the h5ad file which is going to be annotated.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
h5ad_path = "/mnt/workspace_stud/allstud/wp1/data/multiple_cluster.h5ad"

Load anndata and list all possible settings.

In [ ]:
adata = sc.read_h5ad(h5ad_path)
annot.list_possible_settings(repo_path, adata=adata)

Enter general annotation settings.

In [ ]:
# Taxonomy ID or Organism Name
# e.g., "human" or 9606
organism = None

# Column in .obs table where ranked genes groups are stored
# e.g., "rank_genes_groups"
# Enter None if no ranking has been performed yet
rank_genes_column = None

# Column in .var table where gene symbols or Ensembl IDs are stored
# Enter None if the index column of the .var table already has gene symbols or Ensembl IDs
# that you want to use for your annotation
genes_column = None

# The .obs table column of the clustering you want to annotate (e.g., "leiden" or "louvain")
# If None, you can pick one interactively 
clustering_column = None

# Specify whether your index of the .var tables are Ensembl IDs (True) or gene symbols (False)
ensembl = mr.check_ensembl(adata)

# Name of the column to add with the final cell type annotation
# If None, all annotation columns will be kept
celltype_column_name = None

# Whether to delete the created marker lists after annotation or not
delete_lists = True

Specify Marker Lists for Annotation using column specific terms:

- `key`: Specify the column to search in. Use `None` to search across all columns. 
  - Example columns include `Source`, `Organism name`, etc.
- `value`: Define your search terms. Use `-` to exclude keywords and `+` to ensure the keyword must be present.
  - Separate multiple keywords with a comma. For example: `["+panglao.se", "+mouse"]` to include lists from 'panglao.se' and related to 'mouse'.

In [ ]:
column_specific_terms={"Source":"panglao", "Organism name":"human"}

Adjust various settings for marker lists, like 'style' or 'file_name', otherwise the default settings will be used. A dictionary corresponds to a marker list.

Example:
```python
settings = [
    {
        "style": "two_column",
        "file_name": "basic_markers"
    },
    {
        "style": "score",
        "column_specific_terms": {
            "Source": "panglao",
            "Tissue": "heart"
        },
        "file_name": "heart_panglao"
    },
    {
        "force_homology": True,
        "file_name": "homology_markers"
    }
]


In [ ]:
mr_parameters = [{"style":"two_column", "file_name":"two_column"},
            {"style":"score", "file_name":"score"}]

Validate general annotation settings and the mr_parameters.

In [ ]:
wrap.validate_settings(settings=mr_parameters, repo_path=repo_path, adata=adata, organism=organism, 
                        rank_genes_column=rank_genes_column, genes_column=genes_column, clustering_column=clustering_column, ensembl=ensembl,
                        column_specific_terms=column_specific_terms)

## Prepare adata

### Set genes to index, if not already done.

In [ ]:
if genes_column:
    adata.var.reset_index(inplace=True)  # remove old index values and save them in the column ['index']
    adata.var.set_index(genes_column, inplace=True)  # set genes as index
    adata.var.index = adata.var.index.astype('str')  # to avoid index being categorical
    adata.var_names_make_unique(join='_')
    
    # update ensembl if gene identifier has changed
    ensembl = mr.check_ensembl(adata)
    
display(adata.var)

## Create suitable marker list(s)

<details>
    <summary>Click here to see/collapse the function description</summary>
    <p><b>Function Call:</b> create_multiple_marker_lists</p>
    <p>This function calls 'create_marker_lists' with multiple parameter sets to create marker lists. It iterates over each dictionary within a list, using its contents to call 'create_marker_lists'. Default values are assigned for any parameters missing from a dictionary, but these can be overridden by individual dictionary entries.</p>
    <p><b>Parameters (excerpt):</b></p>
    <ul>
        <li><b>settings:</b> list of dict, default [{}] - A list of dictionaries where each dictionary contains parameters for a single call to 'create_marker_lists'. Keys should match the parameter names of 'create_marker_lists', and values are the desired values for those parameters.</li>
        <li><b>style:</b> str, default "score" - Determines the style of the marker lists. Available options include "two_column", "score", "ui", and "panglao".</li>
        <li><b>force_homology:</b> bool, default False - If set to True, the function will attempt to create marker lists via homology, even if marker lists for the given organism already exist.</li>
        <li><b>show_lists:</b> bool, default True - If True, the function displays the marker lists of the query post-creation.</li>
        <li><b>column_specific_terms:</b> dict, default None - A dictionary with column names as keys and lists of search terms as values. If provided, this overrides the 'col_to_search' and 'search_terms' parameters.</li>
        <li><b>adata:</b> AnnData, default None - If provided, the function adds the marker list IDs to the .uns table of the AnnData object.</li>
    </ul>
    <p><b>Returns:</b></p>
    <ul>
        <li><b>list of str:</b> A list of all paths to the created marker lists.</li>
    </ul>
</details>


The paths of the marker lists will be stored in the <b>marker_lists</b> variable. They will work as input for the actual cell type annotation of the next cell.

In [ ]:
marker_lists = wrap.create_multiple_marker_lists(settings=mr_parameters, repo_path=repo_path, organism=organism, 
                                                 ensembl=ensembl, column_specific_terms=column_specific_terms,
                                                 show_lists=True, adata=adata)

## Annotate adata using the created list(s)

<details>
    <summary>Click here to see/collapse the function description</summary>
    <p><b>Function Call:</b> run_annotation</p>
    <p>This function performs annotations on single cell data and allows the user to choose between different annotation methods. It supports both MarkerRepo and SCSA annotations and handles various other parameters related to single cell data annotation.</p>
    <p><b>Parameters:</b></p>
    <ul>
        <li><b>adata:</b> AnnData - The AnnData object to annotate.</li>
        <li><b>marker_repo:</b> bool, default True - Indicates whether to use Marker Repo annotation.</li>
        <li><b>SCSA:</b> bool, default True - Specifies whether to use SCSA annotation.</li>
        <li><b>marker_lists:</b> list of str, default [] - Paths to marker list files for annotation.</li>
        <li><b>mr_obs:</b> str, default "mr" - The .obs key for Marker Repo annotation.</li>
        <li><b>scsa_obs:</b> str, default "scsa" - The .obs key for SCSA annotation.</li>
        <li><b>rank_genes_column:</b> str, default None - The column in .uns containing rank genes scores. If None, the ranking will be performed on the clustering_column.</li>
        <li><b>clustering_column:</b> str, default None - The column in .obs containing clustering information.</li>
        <li><b>reference_obs:</b> str, default None - A reference annotation for comparison, already present in .obs.</li>
        <li><b>keep_all:</b> bool, default False - If True, retains all annotation columns; if False, keeps only the selected annotation column and reference_obs.</li>
        <li><b>verbose:</b> bool, default False - If True, prints additional information during the function execution.</li>
        <li><b>show_ct_tables:</b> bool, default False - If True, displays tables of the MarkerRepo annotation results.</li>
        <li><b>show_plots:</b> bool, default False - If True, displays UMAP plots related to the annotation.</li>
        <li><b>show_comparison:</b> bool, default False - If True, shows a comparison table of the different annotations.</li>
        <li><b>ignore_overwrite:</b> bool, default False - If True, overwrites existing files without confirmation.</li>
        <li><b>celltype_column_name:</b> str, default None - Names the selected cell type annotation column; if None, retains all annotation columns.</li>
    </ul>
</details>


In [ ]:
wrap.run_annotation(adata, SCSA=False, marker_lists=marker_lists, reference_obs=None, show_comparison=True,
                    clustering_column=clustering_column, rank_genes_column=rank_genes_column, 
                    ignore_overwrite=False, verbose=False, show_plots=True, show_ct_tables=True, 
                    celltype_column_name=celltype_column_name)

Show new annotation column.

In [ ]:
adata.obs

In [ ]:
adata.uns["MarkerRepo"]

Delete created marker lists.

In [ ]:
if delete_lists:
    mr.delete_files(marker_lists)
